# Mamba vs Transformer SAE Analysis
state vs delta; Mamba-1.4B (layers 12/24/36) and Pythia-1.4B (layers 6/12/18)

`/raid/sasaki/runs` にある抽出済み・学習済み SAE を総合解析するノート。章立ては:
1. Setup & Data loading / sanity
2. Activations & SAE recon/sparsity (基本統計)
3. Feature-wise stats (dead units, norms)
4. Token/context alignment (concept analysis)
5. Dictionary geometry & cross-condition alignment
6. Cross-layer/model/signal summary
7. Causal eval: ablation & steering (雛形)
8. Robustness checks (雛形)


In [ ]:

import sys, os, json, glob, math
from pathlib import Path
from typing import Optional, List, Tuple

import torch
import pandas as pd
import matplotlib.pyplot as plt

# プロジェクトルートをモジュールパスに追加
sys.path.append('/workspace')  # コンテナ内

RUN_ROOT = Path('/raid/sasaki/runs')
MODELS = {
    'mamba': {
        'root': RUN_ROOT / 'mamba_1.4b_hf',
        'layers': [12, 24, 36],
        'signals': ['state', 'delta'],
    },
    'pythia': {
        'root': RUN_ROOT / 'pythia_1.4b_deduped',
        'layers': [6, 12, 18],
        'signals': ['state', 'delta'],
    },
}

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# ---------------- helpers ----------------

def paths(model_key: str, layer: int, signal: str) -> Path:
    return MODELS[model_key]['root'] / f'layer_{layer}' / signal


def load_meta(model_key: str, layer: int, signal: str):
    path = paths(model_key, layer, signal) / 'meta.json'
    with open(path) as f:
        return json.load(f)


def load_ckpt(model_key: str, layer: int, signal: str, map_location=None):
    path = paths(model_key, layer, signal) / 'sae_checkpoint.pt'
    return torch.load(path, map_location=map_location or device)


def load_sae(model_key: str, layer: int, signal: str):
    from sae_model import SAE
    ckpt = load_ckpt(model_key, layer, signal)
    sae = SAE(
        input_dim=ckpt['input_dim'],
        hidden_dim=ckpt['hidden_dim'],
        mode=ckpt['mode'],
        l1_lambda=ckpt.get('l1_lambda'),
        k_frac=ckpt.get('k_frac', 0.1),
    ).to(device)
    sae.load_state_dict(ckpt['state_dict'])
    sae.eval()
    return sae, ckpt


def iter_chunks(model_key: str, layer: int, signal: str, max_chunks: Optional[int]=None, map_location=None):
    pat = paths(model_key, layer, signal) / 'chunk_*.pt'
    for i, p in enumerate(sorted(glob.glob(str(pat)))):
        if max_chunks is not None and i >= max_chunks:
            break
        yield i, Path(p), torch.load(p, map_location=map_location or device)


## 1. Setup / sanity check
- meta / ckpt / log / chunk の存在確認
- メタ情報（サンプル数・次元など）をざっと見る

In [ ]:

rows = []
for model_key, cfg in MODELS.items():
    for layer in cfg['layers']:
        for signal in cfg['signals']:
            root = paths(model_key, layer, signal)
            meta_path = root / 'meta.json'
            meta = json.load(open(meta_path)) if meta_path.exists() else {}
            rows.append({
                'model': model_key,
                'layer': layer,
                'signal': signal,
                'meta': meta_path.exists(),
                'ckpt': (root / 'sae_checkpoint.pt').exists(),
                'log': (root / 'train_log.jsonl').exists(),
                'chunks': len(list(root.glob('chunk_*.pt'))),
                'total_tokens': meta.get('total_samples'),
                'd_model': meta.get('d_model'),
            })

df_exists = pd.DataFrame(rows)
df_exists

## 2. Activations & SAE recon / sparsity (train_log)
- 最終 loss / recon_loss / global_sparsity
- sparsity 推移

In [ ]:

def load_train_logs(model_key: str):
    rows = []
    cfg = MODELS[model_key]
    for layer in cfg['layers']:
        for signal in cfg['signals']:
            log_path = paths(model_key, layer, signal) / 'train_log.jsonl'
            if not log_path.exists():
                continue
            with open(log_path) as f:
                for line in f:
                    rec = json.loads(line)
                    rec['model'] = model_key
                    rec['layer'] = layer
                    rec['signal'] = signal
                    rows.append(rec)
    return pd.DataFrame(rows)

df_log = pd.concat([load_train_logs('mamba'), load_train_logs('pythia')], ignore_index=True)

last = df_log.sort_values('step').groupby(['model','layer','signal']).tail(1)
display(last[['model','layer','signal','loss','recon_loss','reg_loss','global_sparsity']])

for model_key in MODELS:
    for signal in MODELS[model_key]['signals']:
        plt.figure(figsize=(6,4))
        for layer in MODELS[model_key]['layers']:
            d = df_log[(df_log.model==model_key) & (df_log.signal==signal) & (df_log.layer==layer)]
            if len(d)==0: continue
            plt.plot(d['step'], d['global_sparsity'], label=f'layer {layer}')
        plt.title(f'{model_key} | {signal} | sparsity over training')
        plt.xlabel('step'); plt.ylabel('global sparsity'); plt.legend(); plt.show()


## 3. Feature-wise stats & dead units
- 発火頻度 p_i, 条件付き平均 |z|
- dead / nearly-dead feature 率
- decoder/encoder ノルム分布

In [ ]:

from collections import defaultdict

def compute_feature_stats(model_key: str, layer: int, signal: str, max_chunks=3):
    sae, ckpt = load_sae(model_key, layer, signal)
    nonzero = None; sum_abs = None; total = 0
    for _, _, x in iter_chunks(model_key, layer, signal, max_chunks=max_chunks):
        x = x.to(device)
        with torch.no_grad():
            x_hat, z = sae(x)
        z = z.detach().cpu()
        if nonzero is None:
            nonzero = (z!=0).sum(dim=0)
            sum_abs = z.abs().sum(dim=0)
        else:
            nonzero += (z!=0).sum(dim=0)
            sum_abs += z.abs().sum(dim=0)
        total += z.shape[0]
    freq = nonzero.float() / total
    mean_abs = sum_abs / nonzero.clamp(min=1)
    return freq, mean_abs

stats = {}
for model_key in MODELS:
    for layer in MODELS[model_key]['layers']:
        for signal in MODELS[model_key]['signals']:
            freq, mean_abs = compute_feature_stats(model_key, layer, signal, max_chunks=3)
            stats[(model_key, layer, signal)] = (freq, mean_abs)
            dead_ratio = (freq < 1e-4).float().mean()
            print(model_key, layer, signal, 'mean freq', float(freq.mean()), 'dead<1e-4', float(dead_ratio))


In [ ]:

# 発火頻度ヒストグラム
for model_key in MODELS:
    for signal in MODELS[model_key]['signals']:
        plt.figure(figsize=(6,4))
        for layer in MODELS[model_key]['layers']:
            freq, _ = stats[(model_key, layer, signal)]
            plt.hist(freq.numpy(), bins=50, alpha=0.5, label=f'layer {layer}')
        plt.yscale('log')
        plt.xlabel('feature firing frequency'); plt.ylabel('count (log)')
        plt.title(f'{model_key} | {signal} | firing frequency')
        plt.legend(); plt.show()


In [ ]:

# decoder/encoder 列ノルム分布
for model_key in MODELS:
    for signal in MODELS[model_key]['signals']:
        for layer in MODELS[model_key]['layers']:
            sae, _ = load_sae(model_key, layer, signal)
            dec_norm = sae.decoder.weight.norm(dim=0).detach().cpu()
            enc_norm = sae.encoder.weight.norm(dim=1).detach().cpu()
            plt.figure(figsize=(6,3))
            plt.hist(dec_norm.numpy(), bins=50, alpha=0.6, label='decoder')
            plt.hist(enc_norm.numpy(), bins=50, alpha=0.6, label='encoder')
            plt.title(f'{model_key} | {signal} | layer {layer} | weight norms')
            plt.legend(); plt.show()


## 4. Reconstruction vs Sparsity (sample-wise)
- k(x)=活性ユニット数 と再構成誤差の散布図
- k(x) 分布（箱ひげなど）

In [ ]:

import random

def sample_recon_stats(model_key: str, layer: int, signal: str, max_chunks=1, max_samples=5000):
    sae, ckpt = load_sae(model_key, layer, signal)
    errs = []; ks = []
    taken = 0
    for _, _, x in iter_chunks(model_key, layer, signal, max_chunks=max_chunks):
        x = x.to(device)
        with torch.no_grad():
            x_hat, z = sae(x)
        err = ((x - x_hat)**2).sum(dim=1).sqrt().cpu()
        k = (z!=0).sum(dim=1).cpu()
        errs.append(err); ks.append(k)
        taken += x.shape[0]
        if taken >= max_samples:
            break
    err = torch.cat(errs)[:max_samples]
    k = torch.cat(ks)[:max_samples]
    return err, k

for model_key in MODELS:
    for signal in MODELS[model_key]['signals']:
        plt.figure(figsize=(6,4))
        for layer in MODELS[model_key]['layers']:
            err, k = sample_recon_stats(model_key, layer, signal, max_chunks=1, max_samples=4000)
            plt.scatter(k.numpy(), err.numpy(), s=3, alpha=0.3, label=f'layer {layer}')
        plt.xlabel('active features per sample'); plt.ylabel('reconstruction L2')
        plt.title(f'{model_key} | {signal} | recon vs sparsity')
        plt.legend(); plt.show()


## 5. Dictionary geometry
- decoder コヒーレンス（|cos| 分布）
- PCA との比較のための雛形
- 辞書アラインメント (state/delta, model間, layer間)

In [ ]:

import numpy as np

def decoder_matrix(model_key, layer, signal):
    sae, _ = load_sae(model_key, layer, signal)
    W = sae.decoder.weight.detach().cpu()  # [d_out, d_in]
    return W.T  # [m, d]

# コヒーレンス分布
for model_key in MODELS:
    for signal in MODELS[model_key]['signals']:
        plt.figure(figsize=(6,4))
        all_sims = []
        for layer in MODELS[model_key]['layers']:
            W = decoder_matrix(model_key, layer, signal)
            idx = torch.randperm(W.shape[0])[:256]
            V = W[idx]
            Vn = V / V.norm(dim=1, keepdim=True)
            sims = (Vn @ Vn.T).abs()
            mask = ~torch.eye(sims.shape[0], dtype=bool)
            all_sims.append(sims[mask])
        sims_cat = torch.cat(all_sims)
        plt.hist(sims_cat.numpy(), bins=50, alpha=0.7)
        plt.xlabel('|cos| between decoder atoms'); plt.ylabel('count')
        plt.title(f'{model_key} | {signal} | decoder coherence (sampled)')
        plt.show()


In [ ]:

# state vs delta 辞書アラインメント (同層)
for model_key in MODELS:
    for layer in MODELS[model_key]['layers']:
        W_s = decoder_matrix(model_key, layer, 'state')
        W_d = decoder_matrix(model_key, layer, 'delta')
        W_s_n = W_s / W_s.norm(dim=1, keepdim=True)
        W_d_n = W_d / W_d.norm(dim=1, keepdim=True)
        sims = W_s_n @ W_d_n.T
        max_s = sims.max(dim=1).values
        max_d = sims.max(dim=0).values
        print(f'{model_key} layer {layer}: state->delta mean max {float(max_s.mean()):.3f}, delta->state {float(max_d.mean()):.3f}')


### PCA との比較（雛形）
生表現 `x` の PCA を計算し、辞書列を射影して説明分散を見る。計算コストが大きい場合はサンプルと次元を絞ってください。

In [ ]:

# サンプルして PCA する雛形（必要なとき有効化）
# from sklearn.decomposition import PCA
# def pca_coverage(model_key, layer, signal, n_components=256, max_chunks=1):
#     xs = []
#     for _, _, x in iter_chunks(model_key, layer, signal, max_chunks=max_chunks, map_location='cpu'):
#         xs.append(x)
#     X = torch.cat(xs).numpy()
#     pca = PCA(n_components=n_components).fit(X)
#     W = decoder_matrix(model_key, layer, signal).numpy()
#     proj = W @ pca.components_.T
#     var_expl = (proj**2).sum(axis=1)
#     return pca.explained_variance_ratio_.sum(), var_expl.mean()


## 6. Cross-layer / cross-model / cross-signal summary
- これまでの指標を表やマルチパネルでまとめる（必要に応じて埋めてください）。

In [ ]:

# 例: mean firing freq を一覧表示
rows = []
for k, v in stats.items():
    model_key, layer, signal = k
    freq, mean_abs = v
    rows.append({
        'model': model_key,
        'layer': layer,
        'signal': signal,
        'mean_freq': float(freq.mean()),
        'mean_amp_cond': float(mean_abs.mean()),
        'dead_ratio_<1e-4': float((freq<1e-4).float().mean()),
    })
summary = pd.DataFrame(rows)
display(summary)


## 7. Causal evaluation (ablation / steering) [template]
- forward hook で SAE を挟み、指定 feature を 0 / スケール。
- CE/perplexity や特定トークン logit の変化を測定。
- 実験内容に合わせてこのセルにコードを追加してください。

In [ ]:

# ここに ablation / steering 実験コードを追記するスペース


## 8. Robustness checks [template]
- chunk サブサンプリングで統計のばらつきを確認
- 複数シードで学習した SAE があれば、辞書アラインメントと指標の再現性をチェック


In [ ]:

# ここにロバストネス確認コードを追記
